# Baseline de Transcrição (ASR)

Notebook para validar transcrição dos áudios do CHiME6 usando Whisper (via Transformers), em modo por faixas para evitar uso massivo de memória.

Recorte definido:
- `S02` inteira (arquivo escolhido no setup)
- `S09`: primeira 1h para treino/val; restante para teste

In [ ]:
from pathlib import Path
import json

from transcription import (
    build_transcriber,
    get_session_files,
    split_s09_ranges,
    transcribe_range,
    save_transcription_json,
)

AUDIO_DIR = Path("audio/dev")
OUTPUT_DIR = Path("results/transcriptions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Baseline rápido para validar pipeline.
# Depois você pode trocar para "openai/whisper-small" para mais qualidade.
MODEL_ID = "openai/whisper-small.en"
LANGUAGE = "en"
OUTPUT_SUFFIX =  f"-{MODEL_ID}".replace("/", "_") + ".json"
# Chunk de inferência (segundos). 30 é um bom baseline para estabilidade x velocidade.
CHUNK_SEC = 30

# batch_size: batch interno da pipeline
# CHUNK_BATCH_SIZE: quantidade de chunks enviados por chamada da pipeline
BATCH_SIZE = 8
CHUNK_BATCH_SIZE = 8

## 1. Escolher arquivos de entrada

In [12]:
s02_candidates = get_session_files(AUDIO_DIR, "S02", contains="U01.CH1")
s09_candidates = get_session_files(AUDIO_DIR, "S09", contains="U01.CH1")

if not s02_candidates or not s09_candidates:
    raise FileNotFoundError("Não encontrei os arquivos U01.CH1 para S02/S09 em audio/dev")

S02_WAV = s02_candidates[0]
S09_WAV = s09_candidates[0]

print("S02:", S02_WAV)
print("S09:", S09_WAV)

S02: audio\dev\S02_U01.CH1.wav
S09: audio\dev\S09_U01.CH1.wav


## 2. Inicializar transcritor

In [13]:
asr, generate_kwargs = build_transcriber(model_id=MODEL_ID, language=LANGUAGE)
print("Model loaded:", MODEL_ID)
print("Generate kwargs:", generate_kwargs)

Loading weights: 100%|██████████| 479/479 [00:04<00:00, 105.37it/s]


Model loaded: openai/whisper-small.en
Generate kwargs: {}


## 3. Smoke test (trecho curto)

Primeiro teste em 90s da S02 para validar pipeline sem custo alto.

In [ ]:
smoke = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=S02_WAV,
    start_sec=0.0,
    end_sec=90.0,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

print("Texto (inicio):")
print(smoke.text[:1200])


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Texto (inicio):
P05. Vancouver Stanley Park is 10% larger than Central Park, New York. One of the most powerful, popular ways to explore it is during a bike ride or walk along famous sea wall. Hi, I'm participant P06. Vancouver Stanley Park is 10% larger than Central Park, New York. one of the most popular r- ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P07. Thank you for standing by the 10% margin in Central Park, New York. One of the most popular ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P08, Vancouver Stanley Park, it's 10% more... One of the most popular ways to explore it is during bike ride or along the famous sea wall. Okay, let's see lunch. Okay, here's the pie. Can I help with anything? Yeah, that looks good. Hey, did you put the oven eating or did you put it out? No, I left it on it's pretty


In [19]:
filename_smoke = "S02_smoke_0_90s" + OUTPUT_SUFFIX
save_transcription_json(smoke, OUTPUT_DIR / filename_smoke)
print("Salvo em", OUTPUT_DIR / filename_smoke)

Salvo em results\transcriptions\S02_smoke_0_90s-openai_whisper-small.en.json


## 4. Transcrever S02 inteira

Descomente para rodar completo (pode demorar bastante).

In [ ]:
s02_full = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=S02_WAV,
    start_sec=0.0,
    end_sec=None,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is

In [21]:
filename_full = "S02_full" + OUTPUT_SUFFIX
save_transcription_json(s02_full, OUTPUT_DIR / filename_full)
print("S02 full salvo em", OUTPUT_DIR / filename_full)

S02 full salvo em results\transcriptions\S02_full-openai_whisper-small.en.json


## 5. Transcrever S09 com split train_val/test

In [18]:
ranges = split_s09_ranges()
print(ranges)  # {'train_val': (0.0, 3600.0), 'test': (3600.0, None)}

{'train_val': (0.0, 3600.0), 'test': (3600.0, None)}


In [ ]:
s09_train_val = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=S09_WAV,
    start_sec=ranges['train_val'][0],
    end_sec=ranges['train_val'][1],
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

In [ ]:
filename_s09_val = "S09_val_first_hour" + OUTPUT_SUFFIX
save_transcription_json(s09_train_val, OUTPUT_DIR / filename_s09_val)
print("S09 train_val salvo")

In [ ]:
s09_test = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=S09_WAV,
    start_sec=ranges['test'][0],
    end_sec=ranges['test'][1],
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

In [ ]:
filename_s09_test = "S09_test_rest" + OUTPUT_SUFFIX
save_transcription_json(s09_test, OUTPUT_DIR / filename_s09_test)
print("S09 test salvo")

## 6. Inspecionar JSON salvo

In [23]:
sample_json = OUTPUT_DIR / filename_smoke
if sample_json.exists():
    with open(sample_json, encoding="utf-8") as f:
        data = json.load(f)
    print("n_segments:", len(data.get("segments", [])))
    print("text preview:")
    print(data.get("text", "")[:1000])
else:
    print("Rode a célula de smoke test primeiro.")

n_segments: 15
text preview:
P05. Vancouver Stanley Park is 10% larger than Central Park, New York. One of the most powerful, popular ways to explore it is during a bike ride or walk along famous sea wall. Hi, I'm participant P06. Vancouver Stanley Park is 10% larger than Central Park, New York. one of the most popular r- ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P07. Thank you for standing by the 10% margin in Central Park, New York. One of the most popular ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P08, Vancouver Stanley Park, it's 10% more... One of the most popular ways to explore it is during bike ride or along the famous sea wall. Okay, let's see lunch. Okay, here's the pie. Can I help with anything? Yeah, that looks good. Hey, did you put the oven eating or did you put it out? No, I left it on it's pretty


## 7. Avaliação de erro de palavras (WER/CER)

Aqui comparamos a saída ASR com a transcrição de referência da sessão no mesmo intervalo temporal.

Observação: em CHiME6 há sobreposição de speakers. Esta avaliação concatena segmentos por tempo e funciona como baseline acadêmico de comparação.

In [ ]:
from asr_metrics import evaluate_asr_output

REF_DIR = Path("transcriptions/dev")

# Mapeia cada JSON gerado para a sessão de referência correspondente
asr_eval_jobs = [
    (OUTPUT_DIR / filename_smoke, REF_DIR / "S02.json"),
]

results = []
for asr_json, ref_json in asr_eval_jobs:
    if not asr_json.exists():
        print(f"[PULADO] ASR não encontrado: {asr_json}")
        continue
    if not ref_json.exists():
        print(f"[PULADO] Referência não encontrada: {ref_json}")
        continue

    metrics = evaluate_asr_output(asr_json, ref_json)
    metrics["asr_json"] = asr_json.name
    metrics["ref_json"] = ref_json.name
    results.append(metrics)

if results:
    df_asr_metrics = pd.DataFrame(results)
    cols = [
        "asr_json", "ref_json", "duration_sec",
        "wer", "cer", "ref_words", "hyp_words", "ref_chars", "hyp_chars",
    ]
    display(df_asr_metrics[cols].round(4))
else:
    print("Nenhuma avaliação executada.")

In [ ]:
# Exemplos adicionais (descomente conforme gerar os JSONs)
# asr_eval_jobs = [
#     (OUTPUT_DIR / filename_full, REF_DIR / "S02.json"),
#     (OUTPUT_DIR / filename_s09_val, REF_DIR / "S09.json"),
#     (OUTPUT_DIR / filename_s09_test, REF_DIR / "S09.json"),
# ]
#
# for asr_json, ref_json in asr_eval_jobs:
#     print(asr_json.name, evaluate_asr_output(asr_json, ref_json))

## 8. Métricas de classificação (LSTM) antes do WER

Antes de interpretar WER/CER, verifique o desempenho do classificador fala vs ruído.
Isso evita atribuir ao ASR erros que podem vir do front-end de classificação.

In [ ]:
LSTM_METRICS_JSON = Path('results/lstm_asr/metrics_summary.json')

if LSTM_METRICS_JSON.exists():
    with open(LSTM_METRICS_JSON, encoding='utf-8') as f:
        lstm_payload = json.load(f)

    df_lstm = pd.DataFrame(lstm_payload['metrics'])
    display(df_lstm[['split', 'n_records', 'accuracy', 'precision', 'recall', 'f1', 'auroc']].round(4))
else:
    print('Arquivo de métricas LSTM não encontrado. Rode o notebook 04_lstm_training_chime6.ipynb.')